# Krea-2 LoRA trainer  —  Google Colab

Train a **krea2** style/character LoRA on a Colab GPU, then use it on Krea-2 **Turbo** in ComfyUI / AAAFlow.

**Setup (once):**
1. `Runtime ▸ Change runtime type ▸ GPU` — pick **A100** (Colab Pro, best, ~30 min) or **L4 / T4** (set `SMALL_GPU = True` in the train cell).
2. Run the cells top to bottom (▶ on each, or `Runtime ▸ Run all`).
3. In the **Upload** cell, choose a `.zip` of your images (20–200; captions optional).
4. Set your **TRIGGER** word + **LORA_NAME** in the config cell.
5. The last cell downloads `your-lora.safetensors`.

**Then on your PC:** drop the `.safetensors` into
`C:\AAAFlow\ComfyUI_windows_portable\ComfyUI\models\loras\`
(or `C:\AAAFlow\training\krea2\<name>\output\`) and pick it on the **Images** page.

Weights are pulled straight from HuggingFace (fast on Colab): [Krea-2-Raw](https://huggingface.co/krea/Krea-2-Raw) · [Qwen3-VL TE](https://huggingface.co/Comfy-Org/Qwen3-VL) · [Qwen-Image VAE](https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI). Trainer: [musubi-tuner krea2 docs](https://github.com/kohya-ss/musubi-tuner/blob/main/docs/krea2.md).


### 1 · Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

### 2 · Install the trainer (musubi-tuner)

In [ ]:
!git clone -q https://github.com/kohya-ss/musubi-tuner /content/musubi-tuner
%cd /content/musubi-tuner
!pip install -q -e .
!pip install -q bitsandbytes
print("musubi-tuner installed")

### 3 · Download the krea2 weights from HuggingFace (~35 GB, a few min on Colab)

In [ ]:
from huggingface_hub import hf_hub_download
RAW = hf_hub_download('krea/Krea-2-Raw', 'raw.safetensors')
TE  = hf_hub_download('Comfy-Org/Qwen3-VL', 'text_encoders/qwen3vl_4b_bf16.safetensors')
VAE = hf_hub_download('Comfy-Org/Qwen-Image_ComfyUI', 'split_files/vae/qwen_image_vae.safetensors')
print('DiT :', RAW)
print('TE  :', TE)
print('VAE :', VAE)

### 4 · Upload your dataset
Zip your images (and optional `.txt` captions) into one `.zip`, then run this and choose it.

In [ ]:
import os, zipfile, glob
from google.colab import files
os.makedirs('/content/dataset', exist_ok=True)
print('Choose your dataset .zip ...')
up = files.upload()
for fn in up:
    if fn.lower().endswith('.zip'):
        with zipfile.ZipFile(fn) as z: z.extractall('/content/dataset')
        print('extracted', fn)
imgs = [f for f in glob.glob('/content/dataset/**/*', recursive=True) if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]
print(len(imgs), 'images ready')

### 5 · Config — set your trigger word + name

In [ ]:
TRIGGER   = "crayoncapital"   #@param {type:"string"}
LORA_NAME = "crayon-capital"  #@param {type:"string"}
EPOCHS    = 12                #@param {type:"integer"}

import glob, os
imgs = [f for f in glob.glob('/content/dataset/**/*', recursive=True) if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]
assert imgs, "No images in /content/dataset — run the Upload cell first."
for im in imgs:                       # auto-caption with the trigger if no .txt
    t = os.path.splitext(im)[0] + '.txt'
    if not os.path.exists(t): open(t,'w').write(TRIGGER)
IMG_DIR = os.path.dirname(imgs[0])
TOML = '/content/dataset.toml'
open(TOML,'w').write(f'''[general]
resolution = [1024, 1024]
caption_extension = ".txt"
batch_size = 1
enable_bucket = true
bucket_no_upscale = false

[[datasets]]
image_directory = "{IMG_DIR}"
cache_directory = "/content/cache"
num_repeats = 1
''')
print(len(imgs), 'images |', IMG_DIR)

### 6 · Cache latents + text-encoder outputs (one-time, fast)

In [ ]:
%cd /content/musubi-tuner
!python src/musubi_tuner/krea2_cache_latents.py --dataset_config {TOML} --vae {VAE}
!python src/musubi_tuner/krea2_cache_text_encoder_outputs.py --dataset_config {TOML} --text_encoder {TE} --batch_size 1

### 7 · Train
Leave `SMALL_GPU = False` on an **A100**. Set it **True** for **L4 / T4** (adds fp8 + block-swap to fit smaller VRAM).

In [ ]:
SMALL_GPU = False  #@param {type:"boolean"}
import os
os.makedirs('/content/output', exist_ok=True)
extra = "--fp8_base --fp8_scaled --blocks_to_swap 16" if SMALL_GPU else ""
%cd /content/musubi-tuner
!accelerate launch --num_cpu_threads_per_process 1 --mixed_precision bf16 src/musubi_tuner/krea2_train_network.py --dit {RAW} --vae {VAE} --dataset_config {TOML} --sdpa --mixed_precision bf16 --timestep_sampling shift --weighting_scheme none --discrete_flow_shift 2.5 --optimizer_type adamw8bit --learning_rate 1e-4 --gradient_checkpointing --max_data_loader_n_workers 2 --persistent_data_loader_workers --network_module networks.lora_krea2 --network_dim 32 --network_alpha 32 --max_train_epochs {EPOCHS} --save_every_n_epochs 2 --seed 42 {extra} --output_dir /content/output --output_name {LORA_NAME}

### 8 · Download your LoRA
Save it, then put it in `ComfyUI\models\loras\` on your PC.

In [ ]:
import glob
from google.colab import files
outs = sorted(glob.glob(f'/content/output/{LORA_NAME}*.safetensors'))
print('trained:', outs)
if outs: files.download(outs[-1])

---
**Optional — save straight to Google Drive** (survives disconnects): run
```python
from google.colab import drive; drive.mount('/content/drive')
```
then change the train cell's `--output_dir` to `/content/drive/MyDrive/krea2_loras`.